In [46]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# plotting 설정
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

try:
    # 일반 Python 스크립트 실행 시 (__file__이 존재)
    current_path = Path(__file__).resolve()
except NameError:
    # Jupyter Notebook 실행 시 (__file__ 없음)
    current_path = Path().resolve()

# 현재 경로에서 stock_forecast 폴더까지 자동 탐색
for parent in current_path.parents:
    if (parent / "stock_forecast" / "DATA").is_dir():
        stock_forecast_path = parent / "stock_forecast"
        break
else:
    raise ImportError("stock_forecast/DATA 폴더를 찾을 수 없습니다.")

# sys.path에 추가
if str(stock_forecast_path) not in sys.path:
    sys.path.insert(0, str(stock_forecast_path))

print(f"sys.path에 등록된 경로: {stock_forecast_path}")


from datetime import datetime, timedelta
from typing import Iterable

from sklearn.preprocessing import StandardScaler, RobustScaler
from tqdm import tqdm

# 사용자 유틸 함수들
from DATA.stock_invest_function import *
from datetime import datetime, timedelta

# SQLAlchemy
from sqlalchemy import create_engine, text, Table, MetaData
from sqlalchemy.dialects.mysql import insert as mysql_insert

sys.path에 등록된 경로: C:\Users\MetaM\PycharmProjects\stock_forecast


In [61]:
def clean_numeric_data(series, method='drop'):
    """
    숫자 데이터에서 inf, -inf, NaN 값을 처리

    Parameters:
    - series: pandas Series
    - method: 'drop', 'fill_median', 'fill_mean', 'fill_zero'
    """
    # inf, -inf를 NaN으로 변환
    series = series.replace([np.inf, -np.inf], np.nan)

    if method == 'drop':
        return series.dropna()
    elif method == 'fill_median':
        return series.fillna(series.median())
    elif method == 'fill_mean':
        return series.fillna(series.mean())
    elif method == 'fill_zero':
        return series.fillna(0)
    else:
        return series

def save_valuation_to_db(db_info: dict, table_name: str, df: pd.DataFrame):
    """ticker와 날짜를 기준으로 스마트하게 DB에 저장 (같은 날 replace, 다른 날 append)"""
    try:
        from sqlalchemy import create_engine, text  # text import 추가

        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )

        # 새로 저장할 데이터의 고유 ticker와 날짜 확인
        new_ticker = df['ticker'].iloc[0]
        new_dates = df['date'].dt.date.unique()

        print(f"저장할 데이터: ticker={new_ticker}, 날짜={len(new_dates)}개")

        # 기존 테이블에서 해당 ticker의 기존 데이터 확인
        check_query = f"""
        SELECT DISTINCT DATE(date) as date_only
        FROM {table_name}
        WHERE ticker = '{new_ticker}'
        """

        try:
            existing_dates_df = pd.read_sql(check_query, con=engine)
            existing_dates = set(existing_dates_df['date_only'].tolist())
            print(f"기존 데이터: {len(existing_dates)}개 날짜")
        except:
            existing_dates = set()
            print("기존 데이터 없음")

        # 겹치는 날짜와 새로운 날짜 구분
        new_dates_set = set(new_dates)
        overlap_dates = new_dates_set & existing_dates
        new_only_dates = new_dates_set - existing_dates

        print(f"겹치는 날짜: {len(overlap_dates)}개")
        print(f"새로운 날짜: {len(new_only_dates)}개")

        # 겹치는 날짜가 있으면 해당 데이터 삭제
        if overlap_dates:
            # 간단한 방법: 해당 ticker의 모든 데이터 삭제 후 재추가
            delete_query = f"DELETE FROM {table_name} WHERE ticker = '{new_ticker}'"

            with engine.connect() as conn:
                result = conn.execute(text(delete_query))  # text() 함수로 감싸기
                conn.commit()  # 커밋 추가
                print(f"기존 ticker {new_ticker} 데이터 삭제 완료")

        # 모든 새 데이터 추가
        df.to_sql(
            name=table_name,
            con=engine,
            if_exists='append',
            index=False,
            method='multi'
        )

        print(f"✅ {len(df):,}건 데이터가 '{table_name}' 테이블에 저장되었습니다.")

        # 최종 확인
        final_check_query = f"""
        SELECT COUNT(*) as total_count,
               COUNT(DISTINCT DATE(date)) as unique_dates
        FROM {table_name}
        WHERE ticker = '{new_ticker}'
        """

        final_result = pd.read_sql(final_check_query, con=engine)
        total_count = final_result['total_count'].iloc[0]
        unique_dates = final_result['unique_dates'].iloc[0]

        print(f"최종 확인: ticker {new_ticker}의 총 {total_count}건 데이터, {unique_dates}개 날짜")

    except Exception as e:
        print(f"❌ DB 저장 실패: {e}")

In [48]:
# 사용자가 수정할 부분
tic_name = 'A000660'              # 한국 기업 코드 (예: 삼성전자)
hs_code = '854232'               # 외생변수 HS CODE (수출 데이터용)
st_date = '2010-01-01'
end_date = '2025-08-31'
today_date = pd.to_datetime(datetime.today().date())

# 데이터베이스 정보
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

# -----------------------------
# 1단계: 매출 데이터 추출
# -----------------------------
print("=" * 50)
print("1단계: 매출 데이터 추출 시작")
print("=" * 50)

# korea_fs_data 테이블에서 매출 데이터 가져오기
print("매출 데이터 로딩 중...")
fs_df = fetch_table_data(db_info, "korea_fs_data")
fs_df.rename(columns={'Date': 'date'}, inplace=True)

# 매출액 데이터 필터링
target_indicator = '매출액(천원)'
revenue_raw = fs_df[fs_df['indicator'] == target_indicator].copy()

# 특정 기업의 매출 데이터만 추출
revenue_company = revenue_raw[revenue_raw['symbol'] == tic_name].copy()

print(f"대상 기업: {tic_name}")
print(f"원본 매출 데이터: {len(revenue_company)}건")

if len(revenue_company) == 0:
    print(f"해당 기업의 매출 데이터가 없습니다.")
else:
    # 날짜 및 데이터 정리
    revenue_company['date'] = pd.to_datetime(revenue_company['date'])
    revenue_company['value'] = pd.to_numeric(revenue_company['value'], errors='coerce')
    revenue_company = revenue_company.dropna(subset=['value']).sort_values('date')

    # 분기별 정보 추가
    revenue_company['year'] = revenue_company['date'].dt.year
    revenue_company['quarter'] = revenue_company['date'].dt.quarter
    revenue_company['year_quarter'] = revenue_company['year'].astype(str) + 'Q' + revenue_company['quarter'].astype(str)

    # 분기별로 최신 데이터만 유지 (중복 제거)
    revenue_quarterly = revenue_company.groupby(['year', 'quarter']).agg({
        'date': 'last',
        'value': 'last',
        'year_quarter': 'last',
        'symbol': 'last'
    }).reset_index()

    revenue_quarterly = revenue_quarterly.sort_values(['year', 'quarter']).reset_index(drop=True)
    revenue_quarterly['revenue'] = clean_numeric_data(revenue_quarterly['value'], method='fill_median')

    print(f"정리된 분기별 매출 데이터: {len(revenue_quarterly)}건")
    print(f"데이터 기간: {revenue_quarterly['year_quarter'].iloc[0]} ~ {revenue_quarterly['year_quarter'].iloc[-1]}")

    # 최근 5분기 데이터 출력
    recent_data = revenue_quarterly.tail(5)
    print(f"\n최근 5분기 매출 데이터:")
    for _, row in recent_data.iterrows():
        print(f"  {row['year_quarter']}: {row['revenue']/1e6:.1f}억원")

    # 전역 변수로 저장
    revenue_data_extracted = revenue_quarterly.copy()

    print(f"\n매출 데이터가 'revenue_data_extracted' 변수에 저장되었습니다.")

print("=" * 50)
print("1단계: 매출 데이터 추출 완료")
print("=" * 50)

1단계: 매출 데이터 추출 시작
매출 데이터 로딩 중...
✅ 'korea_fs_data' 테이블에서 5902708건의 데이터를 가져왔습니다.
대상 기업: A000660
원본 매출 데이터: 86건
정리된 분기별 매출 데이터: 86건
데이터 기간: 2004Q1 ~ 2025Q2

최근 5분기 매출 데이터:
  2024Q2: 16423.3억원
  2024Q3: 17573.1억원
  2024Q4: 19767.0억원
  2025Q1: 17639.1억원
  2025Q2: 22232.0억원

매출 데이터가 'revenue_data_extracted' 변수에 저장되었습니다.
1단계: 매출 데이터 추출 완료


In [49]:
# -----------------------------
# 2단계: 수출 데이터 추출
# -----------------------------
print("=" * 50)
print("2단계: 수출 데이터 추출 시작")
print("=" * 50)

# korea_monthly_trade_data_forecast 테이블에서 수출 데이터 가져오기
print("수출 데이터 로딩 중...")
export_df = fetch_table_data(db_info, "korea_monthly_trade_data_forecast")

# 특정 HS코드의 수출 데이터만 추출
export_company = export_df[export_df['root_hs_code'] == hs_code].copy()

print(f"대상 HS코드: {hs_code}")
print(f"원본 수출 데이터: {len(export_company)}건")

if len(export_company) == 0:
    print(f"해당 HS코드의 수출 데이터가 없습니다.")
else:
    # 날짜 및 데이터 정리
    export_company['date'] = pd.to_datetime(export_company['date'])
    export_company['expDlr_forecast_12m'] = pd.to_numeric(export_company['expDlr_forecast_12m'], errors='coerce')
    export_company = export_company.dropna(subset=['expDlr_forecast_12m']).sort_values('date')

    # 분기별 정보 추가
    export_company['year'] = export_company['date'].dt.year
    export_company['quarter'] = export_company['date'].dt.quarter
    export_company['year_quarter'] = export_company['year'].astype(str) + 'Q' + export_company['quarter'].astype(str)

    # 완전한 분기만 필터링 (3월, 6월, 9월, 12월 말까지 있는 분기만)
    print("완전한 분기 데이터만 필터링 중...")

    # 각 분기별로 마지막 월이 분기 말인지 확인
    export_company['month'] = export_company['date'].dt.month
    quarter_end_months = {1: 3, 2: 6, 3: 9, 4: 12}  # 각 분기의 마지막 월

    # 분기별로 그룹화하여 마지막 월 확인
    quarter_check = export_company.groupby(['year', 'quarter']).agg({
        'month': 'max',
        'date': 'count'
    }).reset_index()

    # 완전한 분기만 선별 (마지막 월이 분기 말인 경우)
    complete_quarters = []
    for _, row in quarter_check.iterrows():
        expected_end_month = quarter_end_months[row['quarter']]
        if row['month'] == expected_end_month:
            complete_quarters.append((row['year'], row['quarter']))

    print(f"전체 분기: {len(quarter_check)}개")
    print(f"완전한 분기: {len(complete_quarters)}개")

    # 완전한 분기 데이터만 필터링
    complete_quarter_filter = export_company.apply(
        lambda x: (x['year'], x['quarter']) in complete_quarters, axis=1
    )
    export_company_filtered = export_company[complete_quarter_filter].copy()

    # 분기별 수출액 합산
    export_quarterly = export_company_filtered.groupby(['year', 'quarter']).agg({
        'expDlr_forecast_12m': 'sum',
        'date': 'last',
        'year_quarter': 'last',
        'root_hs_code': 'last'
    }).reset_index()

    export_quarterly = export_quarterly.sort_values(['year', 'quarter']).reset_index(drop=True)

    print(f"정리된 분기별 수출 데이터: {len(export_quarterly)}건")
    print(f"데이터 기간: {export_quarterly['year_quarter'].iloc[0]} ~ {export_quarterly['year_quarter'].iloc[-1]}")

    # 최근 5분기 데이터 출력
    recent_data = export_quarterly.tail(5)
    print(f"\n최근 5분기 수출 데이터:")
    for _, row in recent_data.iterrows():
        print(f"  {row['year_quarter']}: {row['expDlr_forecast_12m']/1e6:.1f}백만달러")

    # 전역 변수로 저장
    export_data_extracted = export_quarterly.copy()

    print(f"\n수출 데이터가 'export_data_extracted' 변수에 저장되었습니다.")

print("=" * 50)
print("2단계: 수출 데이터 추출 완료")
print("=" * 50)



2단계: 수출 데이터 추출 시작
수출 데이터 로딩 중...
✅ 'korea_monthly_trade_data_forecast' 테이블에서 235333건의 데이터를 가져왔습니다.
대상 HS코드: 854232
원본 수출 데이터: 236건
완전한 분기 데이터만 필터링 중...
전체 분기: 75개
완전한 분기: 74개
정리된 분기별 수출 데이터: 74건
데이터 기간: 2008Q1 ~ 2026Q2

최근 5분기 수출 데이터:
  2025Q2: 21328.8백만달러
  2025Q3: 25707.2백만달러
  2025Q4: 26353.5백만달러
  2026Q1: 22681.4백만달러
  2026Q2: 28320.7백만달러

수출 데이터가 'export_data_extracted' 변수에 저장되었습니다.
2단계: 수출 데이터 추출 완료


In [50]:
# export_df[export_df['root_hs_code'] == '854232']

In [51]:
# -----------------------------
# 3단계: YoY 성장률 계산 및 SARIMA 예측
# -----------------------------
print("=" * 50)
print("3단계: YoY 성장률 계산 및 SARIMA 예측 시작")
print("=" * 50)

# Step 3-1: 수출 데이터 YoY 성장률 계산
print("Step 3-1: 수출 YoY 성장률 계산 중...")

export_yoy_df = export_data_extracted.copy()

# YoY 성장률 계산 (전년 동기 대비)
export_yoy_df['exog_var'] = np.nan
for i in range(4, len(export_yoy_df)):
    if export_yoy_df.iloc[i-4]['expDlr_forecast_12m'] != 0:
        yoy_rate = (export_yoy_df.iloc[i]['expDlr_forecast_12m'] / export_yoy_df.iloc[i-4]['expDlr_forecast_12m'] - 1) * 100
        export_yoy_df.loc[export_yoy_df.index[i], 'exog_var'] = yoy_rate

# YoY 성장률이 있는 데이터만 선택
export_exog_df = export_yoy_df[['date', 'exog_var']].dropna().copy()

print(f"수출 YoY 데이터: {len(export_exog_df)}건")
print(f"수출 YoY 기간: {export_exog_df.iloc[0]['date'].strftime('%Y-%m')} ~ {export_exog_df.iloc[-1]['date'].strftime('%Y-%m')}")

# Step 3-2: 매출 데이터 내생변수 준비
print("Step 3-2: 매출 내생변수 준비 중...")

revenue_endog_df = revenue_data_extracted[['date', 'revenue']].copy()
revenue_endog_df.rename(columns={'revenue': 'endog_var'}, inplace=True)

print(f"매출 데이터: {len(revenue_endog_df)}건")
print(f"매출 기간: {revenue_endog_df.iloc[0]['date'].strftime('%Y-%m')} ~ {revenue_endog_df.iloc[-1]['date'].strftime('%Y-%m')}")

# Step 3-3: 데이터 결합
print("Step 3-3: 데이터 결합 중...")

# 분기 기준으로 결합
combined_df = pd.merge(
    revenue_endog_df,
    export_exog_df,
    on='date',
    how='outer'
).sort_values('date')

print(f"결합 전 매출: {len(revenue_endog_df)}건, 수출: {len(export_exog_df)}건")
print(f"결합 후: {len(combined_df)}건")

# 데이터 현황 확인
print("\n데이터 현황:")
print(f"전체 기간: {combined_df.iloc[0]['date'].strftime('%Y-%m')} ~ {combined_df.iloc[-1]['date'].strftime('%Y-%m')}")
print(f"매출 데이터 있는 기간: ~{revenue_endog_df.iloc[-1]['date'].strftime('%Y-%m')}")
print(f"수출 데이터 있는 기간: ~{export_exog_df.iloc[-1]['date'].strftime('%Y-%m')}")

# 명확한 변수명으로 저장
final_combined_data = combined_df.copy()
print(f"\n결합된 데이터프레임이 'final_combined_data' 변수에 저장되었습니다.")
print(f"컬럼: {list(final_combined_data.columns)}")

# endog_var가 있는 데이터만으로 예측용 데이터셋 구성
forecast_df = combined_df[combined_df['endog_var'].notna()].copy()
print(f"SARIMA 예측용 데이터: {len(forecast_df)}건")

# Step 3-4: SARIMA 예측 수행
print("Step 3-4: SARIMA 예측 수행 중...")

if len(forecast_df) < 8:
    print(f"예측용 데이터가 부족합니다: {len(forecast_df)}건 (최소 8건 필요)")
else:
    try:
        # SARIMA 라이브러리 import
        from statsmodels.tsa.statespace.sarimax import SARIMAX
        from itertools import product
        import warnings
        warnings.filterwarnings('ignore')

        # 예측용 데이터 준비
        endog = forecast_df['endog_var'].values
        exog = forecast_df['exog_var'].values if forecast_df['exog_var'].notna().any() else None

        # 결측치 처리
        if exog is not None:
            # exog에 결측치가 있으면 보간
            exog_series = pd.Series(exog)
            exog_series = exog_series.interpolate(method='linear').fillna(method='ffill').fillna(method='bfill')
            exog = exog_series.values

        print(f"내생변수(매출): {len(endog)}개")
        print(f"외생변수(수출YoY): {'사용' if exog is not None else '미사용'}")

        # SARIMA 파라미터 그리드 서치 (외생변수 포함)
        p_values = [0, 1, 2]
        d_values = [0, 1]
        q_values = [0, 1, 2]
        P_values = [0, 1]
        D_values = [0, 1]
        Q_values = [0, 1]
        s_value = 4  # 분기 계절성

        best_aic = float('inf')
        best_params = None
        best_model = None

        print("SARIMA 모델 최적화 중 (외생변수 포함)...")

        for p, d, q, P, D, Q in product(p_values, d_values, q_values, P_values, D_values, Q_values):
            try:
                # 파라미터 수 제한
                total_params = p + q + P + Q + 1
                if total_params >= len(endog) * 0.3:
                    continue

                model = SARIMAX(
                    endog,
                    exog=exog,
                    order=(p, d, q),
                    seasonal_order=(P, D, Q, s_value),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                fitted_model = model.fit(disp=False, maxiter=100)

                if np.isfinite(fitted_model.aic) and fitted_model.aic < best_aic:
                    best_aic = fitted_model.aic
                    best_params = (p, d, q, P, D, Q, s_value)
                    best_model = fitted_model

            except Exception:
                continue

        if best_model is None:
            print("최적화 실패, 기본 모델 사용...")
            model = SARIMAX(endog, exog=exog, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0))
            best_model = model.fit(disp=False)
            best_params = (1, 1, 1, 0, 0, 0, 0)

        print(f"최적 SARIMA 파라미터 (외생변수 포함): {best_params}")
        print(f"AIC (외생변수 포함): {best_model.aic:.2f}")

        # 향후 4분기 예측 (외생변수 포함)
        if exog is not None:
            # 미래 외생변수 값 추정 (최근 평균 사용)
            recent_exog = forecast_df['exog_var'].tail(4).mean()
            future_exog = [recent_exog] * 4
            forecast_result = best_model.forecast(steps=4, exog=future_exog)
        else:
            forecast_result = best_model.forecast(steps=4)

        # 외생변수 없이 SARIMA 예측 추가
        print("\n외생변수 없이 SARIMA 예측 수행 중...")

        # 외생변수 없는 모델을 위한 데이터 준비
        endog_no_exog = forecast_df['endog_var'].values

        # 로그 변환 시도 (양수 데이터만)
        use_log_transform = False
        if np.all(endog_no_exog > 0):
            try:
                endog_log = np.log(endog_no_exog)
                use_log_transform = True
                print("로그 변환 적용")
                endog_for_model = endog_log
            except:
                endog_for_model = endog_no_exog
        else:
            endog_for_model = endog_no_exog

        # 더 제한적인 파라미터 범위
        simple_params = [
            (0, 1, 0, 0, 0, 0, 0),  # 랜덤워크
            (1, 1, 0, 0, 0, 0, 0),  # AR(1) + 차분
            (0, 1, 1, 0, 0, 0, 0),  # MA(1) + 차분
            (1, 1, 1, 0, 0, 0, 0),  # ARIMA(1,1,1)
            (1, 0, 0, 0, 0, 0, 0),  # AR(1)
            (0, 0, 1, 0, 0, 0, 0),  # MA(1)
        ]

        best_aic_no_exog = float('inf')
        best_params_no_exog = None
        best_model_no_exog = None

        for params in simple_params:
            p, d, q, P, D, Q, s = params
            try:
                model_no_exog = SARIMAX(
                    endog_for_model,
                    order=(p, d, q),
                    seasonal_order=(P, D, Q, s),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                fitted_model_no_exog = model_no_exog.fit(disp=False, maxiter=50)

                if np.isfinite(fitted_model_no_exog.aic) and fitted_model_no_exog.aic < best_aic_no_exog:
                    best_aic_no_exog = fitted_model_no_exog.aic
                    best_params_no_exog = params
                    best_model_no_exog = fitted_model_no_exog

            except Exception:
                continue

        # 최종 대체 모델
        if best_model_no_exog is None:
            try:
                print("단순 대체 모델 사용 중...")
                model_no_exog = SARIMAX(endog_for_model, order=(0, 1, 0))
                best_model_no_exog = model_no_exog.fit(disp=False, maxiter=30)
                best_params_no_exog = (0, 1, 0, 0, 0, 0, 0)
            except:
                print("SARIMA 모델 완전 실패 - 단순 예측 사용")
                best_model_no_exog = None

        # 예측 수행
        forecast_result_no_exog = None
        if best_model_no_exog is not None:
            try:
                print(f"외생변수 없는 최적 SARIMA 파라미터: {best_params_no_exog}")
                print(f"외생변수 없는 AIC: {best_model_no_exog.aic:.2f}")

                forecast_result_no_exog = best_model_no_exog.forecast(steps=4)

                # 로그 변환했다면 역변환
                if use_log_transform:
                    forecast_result_no_exog = np.exp(forecast_result_no_exog)

            except Exception as e:
                print(f"SARIMA 예측 실패: {e}")
                best_model_no_exog = None

        # SARIMA가 완전 실패한 경우 단순 통계적 예측
        if best_model_no_exog is None or forecast_result_no_exog is None:
            print("단순 통계적 예측 사용")
            # 최근 8분기 평균 성장률 사용
            recent_revenues = endog_no_exog[-8:]
            if len(recent_revenues) >= 4:
                yoy_growth_rates = []
                for i in range(4, len(recent_revenues)):
                    if recent_revenues[i-4] != 0:
                        growth = (recent_revenues[i] / recent_revenues[i-4] - 1)
                        yoy_growth_rates.append(growth)

                if yoy_growth_rates:
                    avg_growth = np.mean(yoy_growth_rates)
                    conservative_growth = avg_growth * 0.8
                else:
                    conservative_growth = 0.02
            else:
                conservative_growth = 0.02

            last_revenue = endog_no_exog[-1]
            forecast_result_no_exog = []
            for i in range(4):
                pred_value = last_revenue * (1 + conservative_growth) ** (i + 1)
                forecast_result_no_exog.append(pred_value)

            forecast_result_no_exog = np.array(forecast_result_no_exog)
            best_params_no_exog = "simple_growth"
            print(f"단순 성장률 예측 (성장률: {conservative_growth*100:.1f}%)")

        # 예측 결과 정리
        last_date = forecast_df.iloc[-1]['date']
        future_dates = []
        for i in range(1, 5):
            future_date = last_date + pd.DateOffset(months=3*i)
            future_date = future_date + pd.offsets.QuarterEnd(0)
            future_dates.append(future_date)

        # 두 예측 결과 모두 포함하여 데이터프레임 생성
        forecast_results_df = pd.DataFrame({
            'date': future_dates,
            'predicted_revenue': forecast_result,
            'predicted_revenue_without_exog': forecast_result_no_exog
        })

        # 분기 정보 추가
        forecast_results_df['year'] = forecast_results_df['date'].dt.year
        forecast_results_df['quarter'] = forecast_results_df['date'].dt.quarter
        forecast_results_df['year_quarter'] = (forecast_results_df['year'].astype(str) + 'Q' +
                                               forecast_results_df['quarter'].astype(str))

        # 결과 출력
        print("\n=" * 40)
        print("SARIMA 예측 결과 (향후 4분기)")
        print("=" * 40)

        last_revenue = forecast_df.iloc[-1]['endog_var']
        print(f"기준 분기: {forecast_df.iloc[-1]['date'].strftime('%Y년 %mQ')}")
        print(f"기준 매출: {last_revenue/1e6:.1f}억원")
        print()

        print("분기별 예측 결과:")
        for _, row in forecast_results_df.iterrows():
            growth_with = (row['predicted_revenue'] / last_revenue - 1) * 100
            print(f"{row['year_quarter']}:")
            print(f"  외생변수 포함: {row['predicted_revenue']/1e6:.1f}억원 ({growth_with:+.1f}%)")

            if 'predicted_revenue_without_exog' in forecast_results_df.columns and not pd.isna(row['predicted_revenue_without_exog']):
                growth_without = (row['predicted_revenue_without_exog'] / last_revenue - 1) * 100
                print(f"  외생변수 미포함: {row['predicted_revenue_without_exog']/1e6:.1f}억원 ({growth_without:+.1f}%)")
            else:
                print(f"  외생변수 미포함: 예측 실패")

        total_forecast_with = forecast_results_df['predicted_revenue'].sum()
        avg_forecast_with = forecast_results_df['predicted_revenue'].mean()
        avg_growth_with = (avg_forecast_with / last_revenue - 1) * 100

        print()
        if 'predicted_revenue_without_exog' in forecast_results_df.columns and forecast_results_df['predicted_revenue_without_exog'].notna().any():
            total_forecast_without = forecast_results_df['predicted_revenue_without_exog'].sum()
            avg_forecast_without = forecast_results_df['predicted_revenue_without_exog'].mean()
            avg_growth_without = (avg_forecast_without / last_revenue - 1) * 100

            print("4분기 총계 비교:")
            print(f"외생변수 포함: {total_forecast_with/1e6:.1f}억원 (평균성장률: {avg_growth_with:+.1f}%)")
            print(f"외생변수 미포함: {total_forecast_without/1e6:.1f}억원 (평균성장률: {avg_growth_without:+.1f}%)")
        else:
            print("4분기 총계:")
            print(f"외생변수 포함: {total_forecast_with/1e6:.1f}억원 (평균성장률: {avg_growth_with:+.1f}%)")
            print(f"외생변수 미포함: 예측 실패")

        # 전역 변수로 저장
        sarima_forecast_results = forecast_results_df.copy()
        sarima_model_info = {
            'model_with_exog': best_model,
            'model_without_exog': best_model_no_exog if best_model_no_exog is not None else None,
            'params_with_exog': best_params,
            'params_without_exog': best_params_no_exog,
            'aic_with_exog': best_model.aic,
            'aic_without_exog': best_model_no_exog.aic if best_model_no_exog is not None else None,
            'use_exog': exog is not None,
            'log_transform_used': use_log_transform
        }

        print(f"\n예측 결과가 'sarima_forecast_results' 변수에 저장되었습니다.")
        print(f"모델 정보가 'sarima_model_info' 변수에 저장되었습니다.")

    except Exception as e:
        print(f"SARIMA 예측 실패: {str(e)}")

print("=" * 50)
print("3단계: YoY 성장률 계산 및 SARIMA 예측 완료")
print("=" * 50)

3단계: YoY 성장률 계산 및 SARIMA 예측 시작
Step 3-1: 수출 YoY 성장률 계산 중...
수출 YoY 데이터: 70건
수출 YoY 기간: 2009-03 ~ 2026-06
Step 3-2: 매출 내생변수 준비 중...
매출 데이터: 86건
매출 기간: 2004-03 ~ 2025-06
Step 3-3: 데이터 결합 중...
결합 전 매출: 86건, 수출: 70건
결합 후: 90건

데이터 현황:
전체 기간: 2004-03 ~ 2026-06
매출 데이터 있는 기간: ~2025-06
수출 데이터 있는 기간: ~2026-06

결합된 데이터프레임이 'final_combined_data' 변수에 저장되었습니다.
컬럼: ['date', 'endog_var', 'exog_var']
SARIMA 예측용 데이터: 86건
Step 3-4: SARIMA 예측 수행 중...
내생변수(매출): 86개
외생변수(수출YoY): 사용
SARIMA 모델 최적화 중 (외생변수 포함)...
최적 SARIMA 파라미터 (외생변수 포함): (2, 1, 2, 1, 1, 1, 4)
AIC (외생변수 포함): 3279.30

외생변수 없이 SARIMA 예측 수행 중...
로그 변환 적용
외생변수 없는 최적 SARIMA 파라미터: (1, 1, 0, 0, 0, 0, 0)
외생변수 없는 AIC: -55.19

=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
SARIMA 예측 결과 (향후 4분기)
기준 분기: 2025년 06Q
기준 매출: 22232.0억원

분기별 예측 결과:
2025Q3:
  외생변수 포함: 23145.1억원 (+4.1%)
  외생변수 미포함: 23240.7억원 (+4.5%)
2025Q4:
  외생변수 포함: 22822.3억원 (+2.7%)
  외생변수 미포함: 23439.4억원 (+5.4%)
2026Q1:
  외생변수 포함: 19548.2억원 (-12.1%)
  외생변수 미포함:

In [52]:
# -----------------------------
# 4단계: 실제+예측 매출 결합 및 TTM 계산
# -----------------------------
print("=" * 50)
print("4단계: 실제+예측 매출 결합 및 TTM 계산 시작")
print("=" * 50)

# Step 4-1: 실제 매출 데이터 준비
print("Step 4-1: 실제 매출 데이터 준비 중...")

# 실제 매출 데이터 (기존 revenue_data_extracted 사용)
actual_revenue_df = revenue_data_extracted[['date', 'revenue']].copy()
actual_revenue_df['date'] = pd.to_datetime(actual_revenue_df['date'])
actual_revenue_df = actual_revenue_df.sort_values('date').reset_index(drop=True)

print(f"실제 매출 데이터: {len(actual_revenue_df)}건")
print(f"실제 매출 기간: {actual_revenue_df['date'].min().strftime('%Y-%m')} ~ {actual_revenue_df['date'].max().strftime('%Y-%m')}")

# Step 4-2: 예측 데이터 준비
print("Step 4-2: 예측 데이터 준비 중...")

# 예측 데이터 (기존 sarima_forecast_results 사용)
forecast_revenue_df = sarima_forecast_results[['date', 'predicted_revenue', 'predicted_revenue_without_exog']].copy()
forecast_revenue_df['date'] = pd.to_datetime(forecast_revenue_df['date'])
forecast_revenue_df = forecast_revenue_df.sort_values('date').reset_index(drop=True)

print(f"예측 데이터: {len(forecast_revenue_df)}건")
print(f"예측 기간: {forecast_revenue_df['date'].min().strftime('%Y-%m')} ~ {forecast_revenue_df['date'].max().strftime('%Y-%m')}")

# Step 4-3: 실제 매출과 예측 매출 결합
print("Step 4-3: 실제 매출과 예측 매출 결합 중...")

# 실제 매출에는 예측값을 NaN으로, 예측 기간에는 실제값을 NaN으로 설정
actual_revenue_df['predicted_revenue'] = np.nan
actual_revenue_df['predicted_revenue_without_exog'] = np.nan

forecast_revenue_df['revenue'] = np.nan

# 컬럼 순서 맞춤
actual_revenue_df = actual_revenue_df[['date', 'revenue', 'predicted_revenue', 'predicted_revenue_without_exog']]
forecast_revenue_df = forecast_revenue_df[['date', 'revenue', 'predicted_revenue', 'predicted_revenue_without_exog']]

# 데이터 결합
combined_revenue_df = pd.concat([actual_revenue_df, forecast_revenue_df], ignore_index=True)
combined_revenue_df = combined_revenue_df.sort_values('date').reset_index(drop=True)

print(f"결합된 데이터: {len(combined_revenue_df)}건")
print(f"전체 기간: {combined_revenue_df['date'].min().strftime('%Y-%m')} ~ {combined_revenue_df['date'].max().strftime('%Y-%m')}")

# Step 4-4: 결합 시계열 생성
print("Step 4-4: 결합 시계열 생성 중...")

# 외생변수 포함 예측과 실제 매출 결합
combined_revenue_df['revenue_with_exog_forecast'] = combined_revenue_df['revenue'].fillna(combined_revenue_df['predicted_revenue'])

# 외생변수 미포함 예측과 실제 매출 결합
combined_revenue_df['revenue_without_exog_forecast'] = combined_revenue_df['revenue'].fillna(combined_revenue_df['predicted_revenue_without_exog'])

print("결합 시계열 생성 완료:")
print(f"  - revenue_with_exog_forecast: 실제매출 + 외생변수포함예측")
print(f"  - revenue_without_exog_forecast: 실제매출 + 외생변수미포함예측")

# Step 4-5: TTM(Trailing Twelve Months) 계산
print("Step 4-5: TTM 계산 중...")

# TTM 계산 함수
def calculate_ttm(series):
    """4분기(12개월) rolling sum 계산"""
    return series.rolling(window=4, min_periods=1).sum()

# TTM 계산
combined_revenue_df['ttm_revenue_with_exog'] = calculate_ttm(combined_revenue_df['revenue_with_exog_forecast'])
combined_revenue_df['ttm_revenue_without_exog'] = calculate_ttm(combined_revenue_df['revenue_without_exog_forecast'])

# 실제 매출만의 TTM도 계산 (비교용)
combined_revenue_df['ttm_actual_revenue'] = calculate_ttm(combined_revenue_df['revenue'])

print("TTM 계산 완료:")
print(f"  - ttm_revenue_with_exog: 외생변수 포함 TTM")
print(f"  - ttm_revenue_without_exog: 외생변수 미포함 TTM")
print(f"  - ttm_actual_revenue: 실제 매출 TTM (비교용)")

# Step 4-6: 데이터 검증 및 요약
print("Step 4-6: 데이터 검증 및 요약 중...")

# 분기 정보 추가
combined_revenue_df['year'] = combined_revenue_df['date'].dt.year
combined_revenue_df['quarter'] = combined_revenue_df['date'].dt.quarter
combined_revenue_df['year_quarter'] = (combined_revenue_df['year'].astype(str) + 'Q' +
                                      combined_revenue_df['quarter'].astype(str))

# 예측 구간 식별
last_actual_date = actual_revenue_df['date'].max()
combined_revenue_df['is_forecast'] = combined_revenue_df['date'] > last_actual_date

print(f"\n데이터 구조:")
print(f"전체 기간: {combined_revenue_df['year_quarter'].iloc[0]} ~ {combined_revenue_df['year_quarter'].iloc[-1]}")
print(f"실제 데이터: {combined_revenue_df[~combined_revenue_df['is_forecast']]['year_quarter'].iloc[-1]}까지")
print(f"예측 데이터: {combined_revenue_df[combined_revenue_df['is_forecast']]['year_quarter'].iloc[0]}부터")

# 최근 데이터 및 예측 요약
print("\n최근 4분기 실제 + 향후 4분기 예측:")
recent_and_forecast = combined_revenue_df.tail(8)

for _, row in recent_and_forecast.iterrows():
    status = "예측" if row['is_forecast'] else "실제"

    print(f"{row['year_quarter']} ({status}):")

    if not row['is_forecast']:
        # 실제 데이터
        print(f"  실제 매출: {row['revenue']/1e6:.1f}억원")
        print(f"  TTM: {row['ttm_actual_revenue']/1e6:.1f}억원")
    else:
        # 예측 데이터
        print(f"  외생변수 포함: {row['revenue_with_exog_forecast']/1e6:.1f}억원 (TTM: {row['ttm_revenue_with_exog']/1e6:.1f}억원)")
        print(f"  외생변수 미포함: {row['revenue_without_exog_forecast']/1e6:.1f}억원 (TTM: {row['ttm_revenue_without_exog']/1e6:.1f}억원)")

# TTM 성장률 계산
print("\nTTM 성장률 분석:")

# 마지막 실제 TTM과 예측 TTM 비교
last_actual_ttm = combined_revenue_df[~combined_revenue_df['is_forecast']]['ttm_actual_revenue'].iloc[-1]
last_forecast_ttm_with = combined_revenue_df[combined_revenue_df['is_forecast']]['ttm_revenue_with_exog'].iloc[-1]
last_forecast_ttm_without = combined_revenue_df[combined_revenue_df['is_forecast']]['ttm_revenue_without_exog'].iloc[-1]

ttm_growth_with = (last_forecast_ttm_with / last_actual_ttm - 1) * 100
ttm_growth_without = (last_forecast_ttm_without / last_actual_ttm - 1) * 100

print(f"현재 실제 TTM: {last_actual_ttm/1e6:.1f}억원")
print(f"향후 4분기 후 예상 TTM (외생변수 포함): {last_forecast_ttm_with/1e6:.1f}억원 ({ttm_growth_with:+.1f}%)")
print(f"향후 4분기 후 예상 TTM (외생변수 미포함): {last_forecast_ttm_without/1e6:.1f}억원 ({ttm_growth_without:+.1f}%)")

# Step 4-7: 결과 저장
print("\nStep 4-7: 결과 저장 중...")

# 전역 변수로 저장
ttm_combined_data = combined_revenue_df.copy()

# 컬럼 정보 출력
print(f"\n최종 데이터프레임 컬럼:")
for i, col in enumerate(ttm_combined_data.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\n결과가 'ttm_combined_data' 변수에 저장되었습니다.")
print(f"총 {len(ttm_combined_data)}건의 데이터 (실제 {len(actual_revenue_df)}분기 + 예측 {len(forecast_revenue_df)}분기)")

# 데이터 품질 체크
print("\n데이터 품질 체크:")
null_counts = ttm_combined_data.isnull().sum()
for col in ['revenue_with_exog_forecast', 'revenue_without_exog_forecast', 'ttm_revenue_with_exog', 'ttm_revenue_without_exog']:
    null_count = null_counts[col]
    print(f"  {col}: 결측치 {null_count}개")

print("=" * 50)
print("4단계: 실제+예측 매출 결합 및 TTM 계산 완료")
print("=" * 50)

4단계: 실제+예측 매출 결합 및 TTM 계산 시작
Step 4-1: 실제 매출 데이터 준비 중...
실제 매출 데이터: 86건
실제 매출 기간: 2004-03 ~ 2025-06
Step 4-2: 예측 데이터 준비 중...
예측 데이터: 4건
예측 기간: 2025-09 ~ 2026-06
Step 4-3: 실제 매출과 예측 매출 결합 중...
결합된 데이터: 90건
전체 기간: 2004-03 ~ 2026-06
Step 4-4: 결합 시계열 생성 중...
결합 시계열 생성 완료:
  - revenue_with_exog_forecast: 실제매출 + 외생변수포함예측
  - revenue_without_exog_forecast: 실제매출 + 외생변수미포함예측
Step 4-5: TTM 계산 중...
TTM 계산 완료:
  - ttm_revenue_with_exog: 외생변수 포함 TTM
  - ttm_revenue_without_exog: 외생변수 미포함 TTM
  - ttm_actual_revenue: 실제 매출 TTM (비교용)
Step 4-6: 데이터 검증 및 요약 중...

데이터 구조:
전체 기간: 2004Q1 ~ 2026Q2
실제 데이터: 2025Q2까지
예측 데이터: 2025Q3부터

최근 4분기 실제 + 향후 4분기 예측:
2024Q3 (실제):
  실제 매출: 17573.1억원
  TTM: 57731.4억원
2024Q4 (실제):
  실제 매출: 19767.0억원
  TTM: 66193.0억원
2025Q1 (실제):
  실제 매출: 17639.1억원
  TTM: 71402.5억원
2025Q2 (실제):
  실제 매출: 22232.0억원
  TTM: 77211.2억원
2025Q3 (예측):
  외생변수 포함: 23145.1억원 (TTM: 82783.3억원)
  외생변수 미포함: 23240.7억원 (TTM: 82878.9억원)
2025Q4 (예측):
  외생변수 포함: 22822.3억원 (TTM: 85838.5억원)
  외생변수 미포함: 23439.4억원 

In [53]:
# -----------------------------
# 5단계: 월별 PSR 추정 및 향후 12개월 PSR 예측 (수정본)
# -----------------------------
print("=" * 50)
print("5단계: 월별 PSR 추정 및 향후 12개월 PSR 예측 시작")
print("=" * 50)

# Step 5-1: 월별 시가총액 데이터 추출
print("Step 5-1: 월별 시가총액 데이터 추출 중...")

# 시가총액 데이터 가져오기 (참고파일 코드 활용)
# market_cap_df = fetch_table_data(db_info, "ks_listed_company_daily_marketcap")
market_cap_df = get_market_cap_by_ticker(db_info, tic_name)

# 특정 ticker의 시가총액 데이터만 추출
# market_cap_filtered = market_cap_df[
#     (market_cap_df['ticker'] == tic_name) &
#     (market_cap_df['indicator'] == '시가총액')
# ].copy()
market_cap_filtered = market_cap_df

if len(market_cap_filtered) == 0:
    print(f"❌ {tic_name} 기업의 시가총액 데이터가 없습니다.")
    exit()

# 날짜 및 데이터 정리
market_cap_filtered['date'] = pd.to_datetime(market_cap_filtered['date'])
market_cap_filtered['market_cap'] = pd.to_numeric(market_cap_filtered['value'], errors='coerce')
market_cap_filtered = market_cap_filtered.dropna(subset=['market_cap']).sort_values('date')

# 월말 기준으로 시가총액 데이터 집계
market_cap_monthly = market_cap_filtered.set_index('date')['market_cap'].resample('M').last().reset_index()
market_cap_monthly = market_cap_monthly.dropna()

print(f"월별 시가총액 데이터: {len(market_cap_monthly)}건")
print(f"시가총액 기간: {market_cap_monthly['date'].min().strftime('%Y-%m')} ~ {market_cap_monthly['date'].max().strftime('%Y-%m')}")

# Step 5-1-2: 시가총액 결측치 보간
print("Step 5-1-2: 시가총액 결측치 보간 중...")

# 월별 연속 날짜 범위 생성
full_date_range = pd.date_range(
    start=market_cap_monthly['date'].min(),
    end=market_cap_monthly['date'].max(),
    freq='M'
)

# 전체 날짜 범위로 reindex (결측치가 NaN으로 표시됨)
market_cap_complete = market_cap_monthly.set_index('date').reindex(full_date_range)
market_cap_complete.index.name = 'date'

# 선형 보간으로 결측치 채우기
market_cap_complete['market_cap'] = market_cap_complete['market_cap'].interpolate(method='linear')

# 앞뒤 끝값으로 남은 결측치 채우기
market_cap_complete['market_cap'] = market_cap_complete['market_cap'].fillna(method='ffill').fillna(method='bfill')

market_cap_monthly = market_cap_complete.reset_index()

print(f"보간 후 시가총액 데이터: {len(market_cap_monthly)}건")

# Step 5-2: 분기별 매출 데이터에서 월별 TTM 계산
print("Step 5-2: 분기별 매출에서 월별 TTM 계산 중...")

# ttm_combined_data에서 실제 데이터만 추출 (is_forecast=False만)
actual_revenue_ttm = ttm_combined_data[ttm_combined_data['is_forecast'] == False].copy()
actual_revenue_ttm = actual_revenue_ttm[['date', 'revenue']].copy()
actual_revenue_ttm['date'] = pd.to_datetime(actual_revenue_ttm['date'])
actual_revenue_ttm = actual_revenue_ttm.sort_values('date').reset_index(drop=True)
actual_revenue_ttm["ttm_revenue"] = actual_revenue_ttm["revenue"].rolling(window=4).sum()
psr_calculation_df = pd.merge(market_cap_monthly, actual_revenue_ttm, on="date", how="left").ffill(limit=2).dropna()

# PSR 계산
psr_calculation_df['psr'] = psr_calculation_df['market_cap'] / ((psr_calculation_df['ttm_revenue'] + 1e-10)*1000)

# 이상치 제거 (PSR이 음수이거나 너무 큰 값)
psr_calculation_df = psr_calculation_df[
    (psr_calculation_df['psr'] > 0) &
    (psr_calculation_df['psr'] < psr_calculation_df['psr'].quantile(0.99))
]

print(f"실제 월별 PSR 계산 완료: {len(psr_calculation_df)}건")
print(f"PSR 기간: {psr_calculation_df['date'].min().strftime('%Y-%m')} ~ {psr_calculation_df['date'].max().strftime('%Y-%m')}")

# PSR 통계 출력
print(f"\nPSR 통계:")
print(f"  평균: {psr_calculation_df['psr'].mean():.2f}")
print(f"  중앙값: {psr_calculation_df['psr'].median():.2f}")
print(f"  최소값: {psr_calculation_df['psr'].min():.2f}")
print(f"  최대값: {psr_calculation_df['psr'].max():.2f}")

# Step 5-4: 데이터 충분성 검증
print("Step 5-4: 데이터 충분성 검증 중...")

if len(psr_calculation_df) < 60:
    print(f"❌ PSR 데이터가 부족합니다: {len(psr_calculation_df)}건 (최소 60건 필요)")
    print("PSR 예측을 중단합니다.")
else:
    print(f"✓ PSR 데이터 충분성 검증 통과: {len(psr_calculation_df)}건")

    # Step 5-5: 향후 12개월 PSR 예측
    print("\nStep 5-5: 향후 12개월 PSR 예측 수행 중...")

    try:
        # SARIMA 라이브러리 import
        from statsmodels.tsa.statespace.sarimax import SARIMAX
        from itertools import product
        import warnings
        warnings.filterwarnings('ignore')

        # PSR 시계열 데이터 준비
        psr_series = psr_calculation_df['psr'].values

        # 로그 변환 시도
        use_log_transform_psr = False
        if np.all(psr_series > 0):
            try:
                psr_log = np.log(psr_series)
                use_log_transform_psr = True
                print("PSR 로그 변환 적용")
                psr_for_model = psr_log
            except:
                psr_for_model = psr_series
        else:
            psr_for_model = psr_series

        # PSR 예측을 위한 SARIMA 파라미터 (월별 데이터이므로 계절성 12)
        psr_params = [
            (0, 1, 0, 0, 0, 0, 0),    # 랜덤워크
            (1, 1, 0, 0, 0, 0, 0),    # AR(1) + 차분
            (0, 1, 1, 0, 0, 0, 0),    # MA(1) + 차분
            (1, 1, 1, 0, 0, 0, 0),    # ARIMA(1,1,1)
            (1, 0, 0, 0, 0, 0, 0),    # AR(1)
            (0, 0, 1, 0, 0, 0, 0),    # MA(1)
            (1, 1, 1, 1, 0, 1, 12),   # 계절성 포함
            (0, 1, 1, 0, 1, 1, 12),   # 계절성 포함
        ]

        best_aic_psr = float('inf')
        best_params_psr = None
        best_model_psr = None

        print("PSR SARIMA 모델 최적화 중...")

        for params in psr_params:
            p, d, q, P, D, Q, s = params
            try:
                model_psr = SARIMAX(
                    psr_for_model,
                    order=(p, d, q),
                    seasonal_order=(P, D, Q, s),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                fitted_model_psr = model_psr.fit(disp=False, maxiter=100)

                if np.isfinite(fitted_model_psr.aic) and fitted_model_psr.aic < best_aic_psr:
                    best_aic_psr = fitted_model_psr.aic
                    best_params_psr = params
                    best_model_psr = fitted_model_psr

            except Exception:
                continue

        # 최종 대체 모델
        if best_model_psr is None:
            try:
                print("PSR 단순 대체 모델 사용 중...")
                model_psr = SARIMAX(psr_for_model, order=(0, 1, 0))
                best_model_psr = model_psr.fit(disp=False, maxiter=30)
                best_params_psr = (0, 1, 0, 0, 0, 0, 0)
            except:
                print("PSR SARIMA 모델 완전 실패 - 단순 예측 사용")
                best_model_psr = None

        # PSR 예측 수행
        psr_forecast = None
        if best_model_psr is not None:
            try:
                print(f"PSR 최적 SARIMA 파라미터: {best_params_psr}")
                print(f"PSR AIC: {best_model_psr.aic:.2f}")

                psr_forecast = best_model_psr.forecast(steps=12)

                # 로그 변환했다면 역변환
                if use_log_transform_psr:
                    psr_forecast = np.exp(psr_forecast)

            except Exception as e:
                print(f"PSR SARIMA 예측 실패: {e}")
                best_model_psr = None

        # SARIMA 실패 시 단순 통계적 예측
        if best_model_psr is None or psr_forecast is None:
            print("PSR 단순 통계적 예측 사용")
            # 최근 12개월 평균 사용
            recent_psr = psr_series[-12:]
            avg_psr = np.mean(recent_psr)
            psr_forecast = np.array([avg_psr] * 12)
            best_params_psr = "simple_average"
            print(f"평균 PSR 예측: {avg_psr:.2f}")

        # 미래 날짜 생성 (현재 데이터가 있는 마지막 월 다음 달부터)
        last_actual_date = psr_calculation_df['date'].max()
        print(f"실제 PSR 계산 마지막 날짜: {last_actual_date.strftime('%Y-%m')}")

        future_dates_psr = pd.date_range(
            start=last_actual_date + pd.DateOffset(months=1),
            periods=12,
            freq='M'
        )

        print(f"PSR 예측 시작: {future_dates_psr[0].strftime('%Y-%m')}")
        print(f"PSR 예측 종료: {future_dates_psr[-1].strftime('%Y-%m')}")

        # PSR 예측 결과 데이터프레임
        psr_forecast_df = pd.DataFrame({
            'date': future_dates_psr,
            'predicted_psr': psr_forecast
        })

        # 년월 정보 추가
        psr_forecast_df['year'] = psr_forecast_df['date'].dt.year
        psr_forecast_df['month'] = psr_forecast_df['date'].dt.month
        psr_forecast_df['year_month'] = psr_forecast_df['date'].dt.strftime('%Y-%m')

        # Step 5-6: 결과 출력
        print("\n" + "=" * 40)
        print("향후 12개월 PSR 예측 결과")
        print("=" * 40)

        current_psr = psr_calculation_df['psr'].iloc[-1]
        print(f"현재 PSR: {current_psr:.2f}")
        print(f"예측 기간: {psr_forecast_df['year_month'].iloc[0]} ~ {psr_forecast_df['year_month'].iloc[-1]}")
        print()

        print("월별 PSR 예측:")
        for _, row in psr_forecast_df.iterrows():
            change = (row['predicted_psr'] / current_psr - 1) * 100
            print(f"{row['year_month']}: {row['predicted_psr']:.2f} ({change:+.1f}%)")

        avg_predicted_psr = psr_forecast_df['predicted_psr'].mean()
        avg_change = (avg_predicted_psr / current_psr - 1) * 100

        print()
        print(f"평균 예측 PSR: {avg_predicted_psr:.2f}")
        print(f"평균 변화율: {avg_change:+.1f}%")

        # Step 5-7: 전체 PSR 시계열 결합
        print("\nStep 5-7: PSR 시계열 결합 중...")

        # 실제 PSR 데이터에 예측 구분 추가
        psr_calculation_df['predicted_psr'] = np.nan
        psr_calculation_df['is_forecast'] = False

        # 예측 PSR 데이터에 실제 구분 추가
        psr_forecast_df['psr'] = np.nan
        psr_forecast_df['market_cap'] = np.nan
        psr_forecast_df['ttm_revenue'] = np.nan
        psr_forecast_df['is_forecast'] = True

        # 컬럼 순서 맞춤
        common_cols = ['date', 'psr', 'predicted_psr', 'is_forecast']
        psr_historical = psr_calculation_df[common_cols + ['market_cap', 'ttm_revenue']]
        psr_future = psr_forecast_df[common_cols + ['year', 'month', 'year_month']]

        # 결합을 위해 공통 컬럼만 사용
        psr_combined = pd.concat([
            psr_calculation_df[common_cols],
            psr_forecast_df[common_cols]
        ], ignore_index=True)

        psr_combined = psr_combined.sort_values('date').reset_index(drop=True)

        # 결합 PSR 시계열 생성
        psr_combined['combined_psr'] = psr_combined['psr'].fillna(psr_combined['predicted_psr'])

        # 전역 변수로 저장
        monthly_psr_data = psr_calculation_df.copy()
        monthly_psr_forecast = psr_forecast_df.copy()
        monthly_psr_combined = psr_combined.copy()

        print(f"PSR 예측 결과가 다음 변수에 저장되었습니다:")
        print(f"  - monthly_psr_data: 실제 월별 PSR 데이터 ({len(monthly_psr_data)}건)")
        print(f"  - monthly_psr_forecast: 향후 12개월 PSR 예측 ({len(monthly_psr_forecast)}건)")
        print(f"  - monthly_psr_combined: 전체 PSR 시계열 ({len(monthly_psr_combined)}건)")

        # 최근 실제 PSR 데이터 샘플 출력
        print("\n최근 실제 PSR 데이터 (샘플):")
        recent_actual = psr_calculation_df.tail(5)
        for _, row in recent_actual.iterrows():
            print(f"  {row['date'].strftime('%Y-%m')}: PSR {row['psr']:.2f} (시총 {row['market_cap']/1e6:.0f}억, TTM {row['ttm_revenue']/1e6:.0f}억)")

    except Exception as e:
        print(f"PSR 예측 실패: {str(e)}")

print("=" * 50)
print("5단계: 월별 PSR 추정 및 향후 12개월 PSR 예측 완료")
print("=" * 50)

5단계: 월별 PSR 추정 및 향후 12개월 PSR 예측 시작
Step 5-1: 월별 시가총액 데이터 추출 중...
✅ A000660 시가총액 3,718건 조회 완료
월별 시가총액 데이터: 129건
시가총액 기간: 2015-01 ~ 2025-09
Step 5-1-2: 시가총액 결측치 보간 중...
보간 후 시가총액 데이터: 129건
Step 5-2: 분기별 매출에서 월별 TTM 계산 중...
실제 월별 PSR 계산 완료: 124건
PSR 기간: 2015-03 ~ 2025-08

PSR 통계:
  평균: 1.98
  중앙값: 1.94
  최소값: 1.06
  최대값: 3.43
Step 5-4: 데이터 충분성 검증 중...
✓ PSR 데이터 충분성 검증 통과: 124건

Step 5-5: 향후 12개월 PSR 예측 수행 중...
PSR 로그 변환 적용
PSR SARIMA 모델 최적화 중...
PSR 최적 SARIMA 파라미터: (1, 0, 0, 0, 0, 0, 0)
PSR AIC: -200.79
실제 PSR 계산 마지막 날짜: 2025-08
PSR 예측 시작: 2025-09
PSR 예측 종료: 2026-08

향후 12개월 PSR 예측 결과
현재 PSR: 2.54
예측 기간: 2025-09 ~ 2026-08

월별 PSR 예측:
2025-09: 2.52 (-0.7%)
2025-10: 2.50 (-1.3%)
2025-11: 2.49 (-1.9%)
2025-12: 2.47 (-2.6%)
2026-01: 2.46 (-3.2%)
2026-02: 2.44 (-3.8%)
2026-03: 2.42 (-4.4%)
2026-04: 2.41 (-5.0%)
2026-05: 2.39 (-5.6%)
2026-06: 2.38 (-6.2%)
2026-07: 2.37 (-6.7%)
2026-08: 2.35 (-7.3%)

평균 예측 PSR: 2.43
평균 변화율: -4.1%

Step 5-7: PSR 시계열 결합 중...
PSR 예측 결과가 다음 변수에 저장되었습니다:
  - monthly_

In [54]:

# -----------------------------
# 6단계: LSTM과 Prophet을 이용한 12개월 PSR 예측
# -----------------------------
print("=" * 50)
print("6단계: LSTM과 Prophet을 이용한 12개월 PSR 예측 시작")
print("=" * 50)

# Step 6-1: 필요한 라이브러리 import 및 데이터 준비
print("Step 6-1: 라이브러리 import 및 데이터 준비 중...")

try:
    # LSTM용 라이브러리
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.optimizers import Adam
    from sklearn.preprocessing import MinMaxScaler
    lstm_available = True
    print("LSTM 라이브러리 로드 완료")
except ImportError as e:
    print(f"LSTM 라이브러리 로드 실패: {e}")
    lstm_available = False

try:
    # Prophet 라이브러리 (설치 필요: pip install prophet)
    from prophet import Prophet
    prophet_available = True
    print("Prophet 라이브러리 로드 완료")
except ImportError as e:
    print(f"Prophet 라이브러리 로드 실패: {e}")
    print("Prophet 설치 명령어: pip install prophet")
    prophet_available = False

# PSR 데이터 준비
psr_data = monthly_psr_data[['date', 'psr']].copy()
psr_data = psr_data.sort_values('date').reset_index(drop=True)

print(f"PSR 시계열 데이터: {len(psr_data)}건")
print(f"데이터 기간: {psr_data['date'].min().strftime('%Y-%m')} ~ {psr_data['date'].max().strftime('%Y-%m')}")

# Step 6-2: LSTM 모델을 이용한 PSR 예측
print("\nStep 6-2: LSTM 모델을 이용한 PSR 예측...")

lstm_forecast = None
lstm_model_info = None

if lstm_available and len(psr_data) >= 24:  # 최소 24개월 데이터 필요
    try:
        print("LSTM 모델 학습 중...")

        # 데이터 전처리
        psr_values = psr_data['psr'].values.reshape(-1, 1)

        # 정규화
        scaler = MinMaxScaler(feature_range=(0, 1))
        psr_scaled = scaler.fit_transform(psr_values)

        # 시계열 데이터셋 생성 함수
        def create_sequences(data, seq_length):
            X, y = [], []
            for i in range(seq_length, len(data)):
                X.append(data[i-seq_length:i, 0])
                y.append(data[i, 0])
            return np.array(X), np.array(y)

        # 시퀀스 길이 설정 (12개월)
        seq_length = min(12, len(psr_scaled) // 3)

        if len(psr_scaled) > seq_length:
            X, y = create_sequences(psr_scaled, seq_length)

            # 학습/검증 데이터 분할
            train_size = int(len(X) * 0.8)
            X_train, X_val = X[:train_size], X[train_size:]
            y_train, y_val = y[:train_size], y[train_size:]

            # LSTM 데이터 형태로 변환
            X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
            X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))

            # LSTM 모델 구성
            model = Sequential([
                LSTM(50, return_sequences=True, input_shape=(seq_length, 1)),
                Dropout(0.2),
                LSTM(50, return_sequences=False),
                Dropout(0.2),
                Dense(25),
                Dense(1)
            ])

            model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

            # 모델 학습
            tf.random.set_seed(42)
            history = model.fit(
                X_train, y_train,
                batch_size=32,
                epochs=50,
                validation_data=(X_val, y_val),
                verbose=0
            )

            # 12개월 예측 수행
            last_sequence = psr_scaled[-seq_length:].reshape(1, seq_length, 1)
            lstm_predictions = []

            for _ in range(12):
                next_pred = model.predict(last_sequence, verbose=0)
                lstm_predictions.append(next_pred[0, 0])

                # 시퀀스 업데이트
                last_sequence = np.roll(last_sequence, -1, axis=1)
                last_sequence[0, -1, 0] = next_pred[0, 0]

            # 역정규화
            lstm_predictions = np.array(lstm_predictions).reshape(-1, 1)
            lstm_forecast = scaler.inverse_transform(lstm_predictions).flatten()

            # 모델 성능 정보
            train_loss = history.history['loss'][-1]
            val_loss = history.history['val_loss'][-1]

            lstm_model_info = {
                'train_loss': train_loss,
                'val_loss': val_loss,
                'seq_length': seq_length,
                'model_params': model.count_params()
            }

            print(f"LSTM 모델 학습 완료")
            print(f"  - 시퀀스 길이: {seq_length}")
            print(f"  - 학습 손실: {train_loss:.6f}")
            print(f"  - 검증 손실: {val_loss:.6f}")
            print(f"  - 모델 파라미터: {model.count_params():,}개")

        else:
            print("LSTM 학습용 데이터 부족")

    except Exception as e:
        print(f"LSTM 예측 실패: {str(e)}")
        lstm_forecast = None

else:
    if not lstm_available:
        print("LSTM 라이브러리가 없어 예측을 건너뜁니다.")
    else:
        print("LSTM 학습용 데이터 부족 (최소 24개월 필요)")

# Step 6-3: Prophet 모델을 이용한 PSR 예측
print("\nStep 6-3: Prophet 모델을 이용한 PSR 예측...")

prophet_forecast = None
prophet_model_info = None

if prophet_available and len(psr_data) >= 24:  # 최소 24개월 데이터 필요
    try:
        print("Prophet 모델 학습 중...")

        # Prophet용 데이터 준비
        prophet_data = psr_data.copy()
        prophet_data.columns = ['ds', 'y']  # Prophet 요구 형식

        # Prophet 모델 설정
        model_prophet = Prophet(
            daily_seasonality=False,
            weekly_seasonality=False,
            yearly_seasonality=True,
            seasonality_mode='multiplicative',
            changepoint_prior_scale=0.1,
            seasonality_prior_scale=0.1
        )

        # 모델 학습
        model_prophet.fit(prophet_data)

        # 미래 12개월 날짜 생성
        future_dates = model_prophet.make_future_dataframe(periods=12, freq='M')

        # 예측 수행
        forecast_prophet = model_prophet.predict(future_dates)

        # 향후 12개월 예측값만 추출
        prophet_forecast = forecast_prophet.tail(12)['yhat'].values

        # 음수 값 방지
        prophet_forecast = np.maximum(prophet_forecast, 0.01)

        # 모델 성능 정보
        prophet_model_info = {
            'changepoints': len(model_prophet.changepoints),
            'seasonality_components': len(model_prophet.seasonalities),
            'data_points': len(prophet_data)
        }

        print(f"Prophet 모델 학습 완료")
        print(f"  - 변화점: {len(model_prophet.changepoints)}개")
        print(f"  - 계절성 성분: {len(model_prophet.seasonalities)}개")
        print(f"  - 학습 데이터: {len(prophet_data)}개")

    except Exception as e:
        print(f"Prophet 예측 실패: {str(e)}")
        prophet_forecast = None

else:
    if not prophet_available:
        print("Prophet 라이브러리가 없어 예측을 건너뜁니다.")
    else:
        print("Prophet 학습용 데이터 부족 (최소 24개월 필요)")

# Step 6-4: 기존 SARIMA 예측과 비교
print("\nStep 6-4: 예측 결과 비교 및 정리...")

# 미래 날짜 생성
last_date = psr_data['date'].max()
future_dates = pd.date_range(
    start=last_date + pd.DateOffset(months=1),
    periods=12,
    freq='M'
)

# 예측 결과 데이터프레임 생성
forecast_comparison_df = pd.DataFrame({
    'date': future_dates,
    'year_month': future_dates.strftime('%Y-%m')
})

# 기존 SARIMA 예측 추가 (5단계에서 생성됨)
if 'monthly_psr_forecast' in globals():
    sarima_forecast = monthly_psr_forecast['predicted_psr'].values
    forecast_comparison_df['sarima_forecast'] = sarima_forecast
else:
    forecast_comparison_df['sarima_forecast'] = np.nan

# LSTM 예측 추가
if lstm_forecast is not None:
    forecast_comparison_df['lstm_forecast'] = lstm_forecast
else:
    forecast_comparison_df['lstm_forecast'] = np.nan

# Prophet 예측 추가
if prophet_forecast is not None:
    forecast_comparison_df['prophet_forecast'] = prophet_forecast
else:
    forecast_comparison_df['prophet_forecast'] = np.nan

# Step 6-5: 앙상블 예측
print("Step 6-5: 앙상블 예측 계산 중...")

# 유효한 예측값들의 평균으로 앙상블
valid_forecasts = []
forecast_methods = []

if not pd.isna(forecast_comparison_df['sarima_forecast']).all():
    valid_forecasts.append(forecast_comparison_df['sarima_forecast'])
    forecast_methods.append('SARIMA')

if not pd.isna(forecast_comparison_df['lstm_forecast']).all():
    valid_forecasts.append(forecast_comparison_df['lstm_forecast'])
    forecast_methods.append('LSTM')

if not pd.isna(forecast_comparison_df['prophet_forecast']).all():
    valid_forecasts.append(forecast_comparison_df['prophet_forecast'])
    forecast_methods.append('Prophet')

if valid_forecasts:
    # 각 방법에 동일한 가중치 부여
    ensemble_forecast = np.mean(valid_forecasts, axis=0)
    forecast_comparison_df['ensemble_forecast'] = ensemble_forecast

    print(f"앙상블 예측 완료 (사용 방법: {', '.join(forecast_methods)})")
else:
    forecast_comparison_df['ensemble_forecast'] = np.nan
    print("앙상블 예측 실패 - 유효한 예측 결과 없음")

# Step 6-6: 결과 출력
print("\n" + "=" * 50)
print("6단계: PSR 예측 결과 비교")
print("=" * 50)

current_psr = psr_data['psr'].iloc[-1]
print(f"현재 PSR: {current_psr:.2f}")
print(f"예측 기간: {forecast_comparison_df['year_month'].iloc[0]} ~ {forecast_comparison_df['year_month'].iloc[-1]}")
print()

print("월별 PSR 예측 비교:")
for _, row in forecast_comparison_df.iterrows():
    print(f"\n{row['year_month']}:")

    if not pd.isna(row['sarima_forecast']):
        change = (row['sarima_forecast'] / current_psr - 1) * 100
        print(f"  SARIMA: {row['sarima_forecast']:.2f} ({change:+.1f}%)")

    if not pd.isna(row['lstm_forecast']):
        change = (row['lstm_forecast'] / current_psr - 1) * 100
        print(f"  LSTM  : {row['lstm_forecast']:.2f} ({change:+.1f}%)")

    if not pd.isna(row['prophet_forecast']):
        change = (row['prophet_forecast'] / current_psr - 1) * 100
        print(f"  Prophet: {row['prophet_forecast']:.2f} ({change:+.1f}%)")

    if not pd.isna(row['ensemble_forecast']):
        change = (row['ensemble_forecast'] / current_psr - 1) * 100
        print(f"  앙상블: {row['ensemble_forecast']:.2f} ({change:+.1f}%)")

# 평균 예측값 비교
print("\n12개월 평균 예측 PSR:")
for method in ['sarima_forecast', 'lstm_forecast', 'prophet_forecast', 'ensemble_forecast']:
    if not pd.isna(forecast_comparison_df[method]).all():
        avg_forecast = forecast_comparison_df[method].mean()
        avg_change = (avg_forecast / current_psr - 1) * 100
        method_name = method.replace('_forecast', '').upper()
        print(f"  {method_name}: {avg_forecast:.2f} ({avg_change:+.1f}%)")

# Step 6-7: 결과 저장
print("\nStep 6-7: 결과 저장 중...")

# 전역 변수로 저장
psr_forecast_comparison = forecast_comparison_df.copy()
psr_model_comparison = {
    'sarima_available': 'monthly_psr_forecast' in globals(),
    'lstm_available': lstm_forecast is not None,
    'prophet_available': prophet_forecast is not None,
    'lstm_model_info': lstm_model_info,
    'prophet_model_info': prophet_model_info,
    'ensemble_methods': forecast_methods,
    'data_points': len(psr_data)
}

print(f"예측 결과가 다음 변수에 저장되었습니다:")
print(f"  - psr_forecast_comparison: 모든 방법의 PSR 예측 비교")
print(f"  - psr_model_comparison: 모델 정보 및 성능 비교")

print("=" * 50)
print("6단계: LSTM과 Prophet을 이용한 12개월 PSR 예측 완료")
print("=" * 50)

6단계: LSTM과 Prophet을 이용한 12개월 PSR 예측 시작
Step 6-1: 라이브러리 import 및 데이터 준비 중...
LSTM 라이브러리 로드 완료
Prophet 라이브러리 로드 완료
PSR 시계열 데이터: 124건
데이터 기간: 2015-03 ~ 2025-08

Step 6-2: LSTM 모델을 이용한 PSR 예측...
LSTM 모델 학습 중...


DEBUG:cmdstanpy:cmd: where.exe tbb.dll
cwd: None
DEBUG:cmdstanpy:TBB already found in load path


LSTM 모델 학습 완료
  - 시퀀스 길이: 12
  - 학습 손실: 0.012690
  - 검증 손실: 0.024074
  - 모델 파라미터: 31,901개

Step 6-3: Prophet 모델을 이용한 PSR 예측...
Prophet 모델 학습 중...


DEBUG:cmdstanpy:input tempfile: C:\Users\MetaM\AppData\Local\Temp\tmpmyzu09zy\cqx78stq.json
DEBUG:cmdstanpy:input tempfile: C:\Users\MetaM\AppData\Local\Temp\tmpmyzu09zy\jmblu8a1.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['C:\\Users\\MetaM\\PycharmProjects\\stock_forecast\\.venv\\Lib\\site-packages\\prophet\\stan_model\\prophet_model.bin', 'random', 'seed=39957', 'data', 'file=C:\\Users\\MetaM\\AppData\\Local\\Temp\\tmpmyzu09zy\\cqx78stq.json', 'init=C:\\Users\\MetaM\\AppData\\Local\\Temp\\tmpmyzu09zy\\jmblu8a1.json', 'output', 'file=C:\\Users\\MetaM\\AppData\\Local\\Temp\\tmpmyzu09zy\\prophet_modelyopy830n\\prophet_model-20250908144621.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
14:46:21 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
14:46:22 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


Prophet 모델 학습 완료
  - 변화점: 25개
  - 계절성 성분: 1개
  - 학습 데이터: 124개

Step 6-4: 예측 결과 비교 및 정리...
Step 6-5: 앙상블 예측 계산 중...
앙상블 예측 완료 (사용 방법: SARIMA, LSTM, Prophet)

6단계: PSR 예측 결과 비교
현재 PSR: 2.54
예측 기간: 2025-09 ~ 2026-08

월별 PSR 예측 비교:

2025-09:
  SARIMA: 2.52 (-0.7%)
  LSTM  : 2.36 (-7.0%)
  Prophet: 2.38 (-6.2%)
  앙상블: 2.42 (-4.6%)

2025-10:
  SARIMA: 2.50 (-1.3%)
  LSTM  : 2.38 (-6.1%)
  Prophet: 2.37 (-6.5%)
  앙상블: 2.42 (-4.6%)

2025-11:
  SARIMA: 2.49 (-1.9%)
  LSTM  : 2.36 (-6.8%)
  Prophet: 2.46 (-3.0%)
  앙상블: 2.44 (-3.9%)

2025-12:
  SARIMA: 2.47 (-2.6%)
  LSTM  : 2.31 (-8.9%)
  Prophet: 2.49 (-2.0%)
  앙상블: 2.42 (-4.5%)

2026-01:
  SARIMA: 2.46 (-3.2%)
  LSTM  : 2.24 (-11.7%)
  Prophet: 2.56 (+1.0%)
  앙상블: 2.42 (-4.6%)

2026-02:
  SARIMA: 2.44 (-3.8%)
  LSTM  : 2.17 (-14.6%)
  Prophet: 2.47 (-2.6%)
  앙상블: 2.36 (-7.0%)

2026-03:
  SARIMA: 2.42 (-4.4%)
  LSTM  : 2.09 (-17.7%)
  Prophet: 2.56 (+0.9%)
  앙상블: 2.36 (-7.1%)

2026-04:
  SARIMA: 2.41 (-5.0%)
  LSTM  : 2.00 (-21.1%)
  Prophet: 2

In [55]:
# -----------------------------
# 7단계: PSR과 TTM 매출을 이용한 가치 추정
# -----------------------------
print("=" * 50)
print("7단계: PSR과 TTM 매출을 이용한 가치 추정 시작")
print("=" * 50)

# Step 7-1: TTM 매출 데이터 준비
print("Step 7-1: TTM 매출 데이터 준비 중...")

# ttm_combined_data에서 TTM 매출 데이터 추출
ttm_revenue_data = ttm_combined_data[['date', 'ttm_revenue_with_exog', 'ttm_revenue_without_exog']].copy()
ttm_revenue_data['date'] = pd.to_datetime(ttm_revenue_data['date'])
ttm_revenue_data = ttm_revenue_data.sort_values('date').reset_index(drop=True)

# 분기 정보 추가
ttm_revenue_data['year'] = ttm_revenue_data['date'].dt.year
ttm_revenue_data['quarter'] = ttm_revenue_data['date'].dt.quarter

print(f"TTM 매출 데이터: {len(ttm_revenue_data)}건")
print(f"TTM 기간: {ttm_revenue_data['date'].min().strftime('%Y-%m')} ~ {ttm_revenue_data['date'].max().strftime('%Y-%m')}")

# Step 7-2: PSR 예측 데이터 준비
print("Step 7-2: PSR 예측 데이터 준비 중...")

# forecast_comparison_df (6단계에서 생성됨)
valuation_df = psr_forecast_comparison.copy()
valuation_df['date'] = pd.to_datetime(valuation_df['date'])

# 분기 정보 추가
valuation_df['year'] = valuation_df['date'].dt.year
valuation_df['month'] = valuation_df['date'].dt.month
valuation_df['quarter'] = valuation_df['date'].dt.quarter

print(f"PSR 예측 데이터: {len(valuation_df)}건")
print(f"예측 기간: {valuation_df['date'].min().strftime('%Y-%m')} ~ {valuation_df['date'].max().strftime('%Y-%m')}")

# Step 7-3: 분기별 TTM 매출 매칭 규칙 적용
print("Step 7-3: 분기별 TTM 매출 매칭 중...")

# TTM 매출 매칭 함수
def get_ttm_revenue_for_quarter(forecast_year, forecast_quarter, ttm_data):
    """
    분기별 TTM 매출 매칭 규칙:
    1분기: 전년도 4분기 TTM
    2분기: 같은년도 1분기 TTM
    3분기: 같은년도 2분기 TTM
    4분기: 같은년도 3분기 TTM
    """

    if forecast_quarter == 1:  # 1분기: 전년도 4분기
        target_year = forecast_year - 1
        target_quarter = 4
    elif forecast_quarter == 2:  # 2분기: 같은년도 1분기
        target_year = forecast_year
        target_quarter = 1
    elif forecast_quarter == 3:  # 3분기: 같은년도 2분기
        target_year = forecast_year
        target_quarter = 2
    elif forecast_quarter == 4:  # 4분기: 같은년도 3분기
        target_year = forecast_year
        target_quarter = 3
    else:
        return None, None

    # 해당 년도/분기의 TTM 매출 찾기
    target_data = ttm_data[
        (ttm_data['year'] == target_year) &
        (ttm_data['quarter'] == target_quarter)
    ]

    if len(target_data) > 0:
        target_row = target_data.iloc[-1]  # 최신 데이터 사용
        return target_row['ttm_revenue_with_exog'], target_row['ttm_revenue_without_exog']
    else:
        return None, None

# TTM 매출 매칭 수행
valuation_df['matched_ttm_with_exog'] = np.nan
valuation_df['matched_ttm_without_exog'] = np.nan

for i, row in valuation_df.iterrows():
    forecast_year = row['year']
    forecast_quarter = row['quarter']

    ttm_with_exog, ttm_without_exog = get_ttm_revenue_for_quarter(
        forecast_year, forecast_quarter, ttm_revenue_data
    )

    if ttm_with_exog is not None:
        valuation_df.loc[i, 'matched_ttm_with_exog'] = ttm_with_exog
        valuation_df.loc[i, 'matched_ttm_without_exog'] = ttm_without_exog

print("TTM 매출 매칭 완료")

# 매칭 결과 확인
matched_count = valuation_df['matched_ttm_with_exog'].notna().sum()
print(f"성공적으로 매칭된 데이터: {matched_count}/{len(valuation_df)}건")

# Step 7-4: 밸류에이션 계산
print("Step 7-4: 밸류에이션 계산 중...")

# PSR 예측 컬럼들
psr_columns = ['sarima_forecast', 'lstm_forecast', 'prophet_forecast', 'ensemble_forecast']

# 밸류에이션 계산 (PSR × TTM 매출)
for psr_col in psr_columns:
    valuation_col = psr_col.replace('_forecast', '_valuation')

    # 외생변수 포함 TTM으로 밸류에이션 계산
    valuation_df[valuation_col] = (
        valuation_df[psr_col] * valuation_df['matched_ttm_with_exog']
    )

print("밸류에이션 계산 완료")

# Step 7-5: 결과 검증 및 요약
print("Step 7-5: 결과 검증 및 요약 중...")

# 유효한 밸류에이션 데이터 확인
valuation_columns = ['sarima_valuation', 'lstm_valuation', 'prophet_valuation', 'ensemble_valuation']

for val_col in valuation_columns:
    valid_count = valuation_df[val_col].notna().sum()
    print(f"{val_col}: {valid_count}/{len(valuation_df)}건 유효")

# Step 7-6: 분기별 매칭 규칙 상세 출력
print("\nStep 7-6: 분기별 매칭 상세 결과...")

print("\n분기별 TTM 매칭 상세:")
for i, row in valuation_df.iterrows():
    forecast_quarter_name = f"{row['year']}Q{row['quarter']}"

    if row['quarter'] == 1:
        reference_quarter = f"{row['year']-1}Q4"
    elif row['quarter'] == 2:
        reference_quarter = f"{row['year']}Q1"
    elif row['quarter'] == 3:
        reference_quarter = f"{row['year']}Q2"
    elif row['quarter'] == 4:
        reference_quarter = f"{row['year']}Q3"

    matched_ttm = row['matched_ttm_with_exog']

    if pd.notna(matched_ttm):
        print(f"{forecast_quarter_name} → {reference_quarter} TTM: {matched_ttm/1e6:.1f}억원")
    else:
        print(f"{forecast_quarter_name} → {reference_quarter} TTM: 매칭 실패")

# Step 7-7: 밸류에이션 결과 출력
print("\n" + "=" * 50)
print("7단계: 기업 가치 추정 결과")
print("=" * 50)

print(f"평가 기간: {valuation_df['year_month'].iloc[0]} ~ {valuation_df['year_month'].iloc[-1]}")
print()

print("월별 기업 가치 추정 (억원):")
for _, row in valuation_df.iterrows():
    print(f"\n{row['year_month']} (Q{row['quarter']}):")

    if pd.notna(row['sarima_valuation']):
        print(f"  SARIMA   : {row['sarima_valuation']/1e6:,.0f}억원")

    if pd.notna(row['lstm_valuation']):
        print(f"  LSTM     : {row['lstm_valuation']/1e6:,.0f}억원")

    if pd.notna(row['prophet_valuation']):
        print(f"  Prophet  : {row['prophet_valuation']/1e6:,.0f}억원")

    if pd.notna(row['ensemble_valuation']):
        print(f"  앙상블   : {row['ensemble_valuation']/1e6:,.0f}억원")

# 평균 기업가치 계산
print("\n12개월 평균 기업 가치:")
for val_col in valuation_columns:
    if valuation_df[val_col].notna().any():
        avg_valuation = valuation_df[val_col].mean()
        method_name = val_col.replace('_valuation', '').upper()
        print(f"  {method_name}: {avg_valuation/1e6:,.0f}억원")

# 현재 시가총액과 비교 (참고용)
print(f"\n참고: 현재 시가총액과의 비교")
try:
    # 최신 시가총액 정보
    latest_market_cap = market_cap_monthly['market_cap'].iloc[-1]
    print(f"현재 시가총액: {latest_market_cap/1e6:,.0f}억원")

    # 앙상블 평균 대비 프리미엄/디스카운트
    if valuation_df['ensemble_valuation'].notna().any():
        avg_ensemble_val = valuation_df['ensemble_valuation'].mean()
        premium_discount = (avg_ensemble_val / latest_market_cap - 1) * 100
        print(f"앙상블 평균 대비: {premium_discount:+.1f}% ({'프리미엄' if premium_discount > 0 else '디스카운트'})")

except:
    print("현재 시가총액 정보를 가져올 수 없습니다.")

# Step 7-8: 결과 저장
print("\nStep 7-8: 결과 저장 중...")

# 전역 변수로 저장
company_valuation_results = valuation_df.copy()

# 요약 통계
valuation_summary = {}
for val_col in valuation_columns:
    if valuation_df[val_col].notna().any():
        method_name = val_col.replace('_valuation', '')
        valuation_summary[method_name] = {
            'avg_valuation': valuation_df[val_col].mean(),
            'min_valuation': valuation_df[val_col].min(),
            'max_valuation': valuation_df[val_col].max(),
            'valid_months': valuation_df[val_col].notna().sum()
        }

print(f"밸류에이션 결과가 다음 변수에 저장되었습니다:")
print(f"  - company_valuation_results: 월별 상세 밸류에이션 결과")
print(f"  - valuation_summary: 방법별 요약 통계")

print(f"\n컬럼 정보:")
for col in company_valuation_results.columns:
    print(f"  - {col}")

print("=" * 50)
print("7단계: PSR과 TTM 매출을 이용한 가치 추정 완료")
print("=" * 50)

7단계: PSR과 TTM 매출을 이용한 가치 추정 시작
Step 7-1: TTM 매출 데이터 준비 중...
TTM 매출 데이터: 90건
TTM 기간: 2004-03 ~ 2026-06
Step 7-2: PSR 예측 데이터 준비 중...
PSR 예측 데이터: 12건
예측 기간: 2025-09 ~ 2026-08
Step 7-3: 분기별 TTM 매출 매칭 중...
TTM 매출 매칭 완료
성공적으로 매칭된 데이터: 12/12건
Step 7-4: 밸류에이션 계산 중...
밸류에이션 계산 완료
Step 7-5: 결과 검증 및 요약 중...
sarima_valuation: 12/12건 유효
lstm_valuation: 12/12건 유효
prophet_valuation: 12/12건 유효
ensemble_valuation: 12/12건 유효

Step 7-6: 분기별 매칭 상세 결과...

분기별 TTM 매칭 상세:
2025Q3 → 2025Q2 TTM: 77211.2억원
2025Q4 → 2025Q3 TTM: 82783.3억원
2025Q4 → 2025Q3 TTM: 82783.3억원
2025Q4 → 2025Q3 TTM: 82783.3억원
2026Q1 → 2025Q4 TTM: 85838.5억원
2026Q1 → 2025Q4 TTM: 85838.5억원
2026Q1 → 2025Q4 TTM: 85838.5억원
2026Q2 → 2026Q1 TTM: 87747.6억원
2026Q2 → 2026Q1 TTM: 87747.6억원
2026Q2 → 2026Q1 TTM: 87747.6억원
2026Q3 → 2026Q2 TTM: 86017.7억원
2026Q3 → 2026Q2 TTM: 86017.7억원

7단계: 기업 가치 추정 결과
평가 기간: 2025-09 ~ 2026-08

월별 기업 가치 추정 (억원):

2025-09 (Q3):
  SARIMA   : 194,552억원
  LSTM     : 182,149억원
  Prophet  : 183,659억원
  앙상블   : 186,786억원

2025-10

In [56]:
# -----------------------------
# 8단계  Long Format DB 데이터 생성 (수정본)
# -----------------------------
print("=" * 50)
print("Long Format DB 데이터 생성 시작")
print("=" * 50)

# Step 1: 데이터 준비 및 검증
print("Step 1: 데이터 준비 및 검증 중...")

# 현재 날짜를 forecast_date로 사용
forecast_date = pd.Timestamp.now().strftime('%Y-%m-%d')
print(f"예측 기준일: {forecast_date}")

# ttm_combined_data에 추가 컬럼 생성
ttm_combined_data['forecast_date'] = forecast_date  # 계산하는 날의 날짜 추가
ttm_combined_data['exog_var'] = hs_code  # hs_code 추가

# ttm_combined_data에서 필요한 컬럼 추출 (추가 컬럼 포함)
ttm_data_subset = ttm_combined_data[[
   'date',
   'revenue_with_exog_forecast',
   'revenue_without_exog_forecast',
   'is_forecast',
   'ttm_revenue_with_exog',
   'ttm_revenue_without_exog',
   'forecast_date',  # 추가
   'exog_var'       # 추가
]].copy()

# company_valuation_results(valuation_df)에서 필요한 컬럼 추출
valuation_data_subset = company_valuation_results[[
   'date',
   'sarima_valuation',
   'prophet_valuation',
   'lstm_valuation',
   'ensemble_valuation',
   'matched_ttm_with_exog',
   'matched_ttm_without_exog'
]].copy()

print(f"TTM 데이터: {len(ttm_data_subset)}건")
print(f"밸류에이션 데이터: {len(valuation_data_subset)}건")

# Step 2: TTM 데이터를 Long Format으로 변환
print("Step 2: TTM 데이터 Long Format 변환 중...")

ttm_long_list = []

for _, row in ttm_data_subset.iterrows():
   date = row['date']

   # 각 indicator별로 레코드 생성 (추가 컬럼 포함)
   items_to_convert = [
       ('revenue_with_exog_forecast', row['revenue_with_exog_forecast']),
       ('revenue_without_exog_forecast', row['revenue_without_exog_forecast']),
       ('is_forecast', 1 if row['is_forecast'] else 0),
       ('ttm_revenue_with_exog', row['ttm_revenue_with_exog']),
       ('ttm_revenue_without_exog', row['ttm_revenue_without_exog']),
       ('forecast_date', row['forecast_date']),  # 추가
       ('exog_var', row['exog_var'])            # 추가
   ]

   for item_name, item_value in items_to_convert:
       if pd.notna(item_value):
           ttm_long_list.append({
               'date': date,
               'ticker': tic_name,
               'indicator': item_name,  # indicator 사용
               'value': item_value
           })

ttm_long_df = pd.DataFrame(ttm_long_list)
print(f"TTM Long Format: {len(ttm_long_df)}개 레코드")

# Step 3: 밸류에이션 데이터를 Long Format으로 변환
print("Step 3: 밸류에이션 데이터 Long Format 변환 중...")

valuation_long_list = []

for _, row in valuation_data_subset.iterrows():
   date = row['date']

   items_to_convert = [
       ('sarima_valuation', row['sarima_valuation']),
       ('prophet_valuation', row['prophet_valuation']),
       ('lstm_valuation', row['lstm_valuation']),
       ('ensemble_valuation', row['ensemble_valuation']),
       ('matched_ttm_with_exog', row['matched_ttm_with_exog']),
       ('matched_ttm_without_exog', row['matched_ttm_without_exog']),
       ('forecast_date', forecast_date),  # 밸류에이션에도 forecast_date 추가
       ('exog_var', hs_code)             # 밸류에이션에도 exog_var 추가
   ]

   for item_name, item_value in items_to_convert:
       if pd.notna(item_value):
           valuation_long_list.append({
               'date': date,
               'ticker': tic_name,
               'indicator': item_name,  # indicator 사용
               'value': item_value
           })

valuation_long_df = pd.DataFrame(valuation_long_list)
print(f"밸류에이션 Long Format: {len(valuation_long_df)}개 레코드")

# Step 4: 두 데이터프레임 결합
print("Step 4: 데이터 결합 중...")

long_format_for_db = pd.concat([ttm_long_df, valuation_long_df], ignore_index=True)
long_format_for_db = long_format_for_db.sort_values(['date', 'indicator']).reset_index(drop=True)  # indicator로 수정

print(f"결합된 Long Format: {len(long_format_for_db)}개 레코드")

# Step 5: 데이터 타입 정리
print("Step 5: 데이터 타입 정리 중...")

long_format_for_db['date'] = pd.to_datetime(long_format_for_db['date'])

def convert_value_type(row):
   if row['indicator'] in ['forecast_date', 'exog_var']:  # indicator로 수정
       return str(row['value'])
   else:
       try:
           return float(row['value'])
       except:
           return row['value']

long_format_for_db['value'] = long_format_for_db.apply(convert_value_type, axis=1)

# Step 6: 데이터 품질 검증
print("Step 6: 데이터 품질 검증 중...")

indicator_counts = long_format_for_db['indicator'].value_counts().sort_index()  # indicator로 수정
print("\nIndicator별 레코드 수:")  # 출력 메시지도 수정
for indicator, count in indicator_counts.items():
   print(f"  {indicator}: {count}개")

date_range = f"{long_format_for_db['date'].min().strftime('%Y-%m-%d')} ~ {long_format_for_db['date'].max().strftime('%Y-%m-%d')}"
print(f"\n날짜 범위: {date_range}")
print(f"Ticker: {tic_name}")
print(f"HS Code: {hs_code}")
print(f"예측 기준일: {forecast_date}")

# Step 7: 결과 저장
print("\nStep 7: 결과 저장...")

long_format_for_db_final = long_format_for_db.copy()

print(f"Long Format DB 데이터가 'long_format_for_db_final' 변수에 저장되었습니다.")
print(f"총 {len(long_format_for_db_final):,}개 레코드")
print(f"컬럼: {list(long_format_for_db_final.columns)}")

print("=" * 50)
print("Long Format DB 데이터 생성 완료")
print("=" * 50)

Long Format DB 데이터 생성 시작
Step 1: 데이터 준비 및 검증 중...
예측 기준일: 2025-09-08
TTM 데이터: 90건
밸류에이션 데이터: 12건
Step 2: TTM 데이터 Long Format 변환 중...
TTM Long Format: 630개 레코드
Step 3: 밸류에이션 데이터 Long Format 변환 중...
밸류에이션 Long Format: 96개 레코드
Step 4: 데이터 결합 중...
결합된 Long Format: 726개 레코드
Step 5: 데이터 타입 정리 중...
Step 6: 데이터 품질 검증 중...

Indicator별 레코드 수:
  ensemble_valuation: 12개
  exog_var: 102개
  forecast_date: 102개
  is_forecast: 90개
  lstm_valuation: 12개
  matched_ttm_with_exog: 12개
  matched_ttm_without_exog: 12개
  prophet_valuation: 12개
  revenue_with_exog_forecast: 90개
  revenue_without_exog_forecast: 90개
  sarima_valuation: 12개
  ttm_revenue_with_exog: 90개
  ttm_revenue_without_exog: 90개

날짜 범위: 2004-03-31 ~ 2026-08-31
Ticker: A000660
HS Code: 854232
예측 기준일: 2025-09-08

Step 7: 결과 저장...
Long Format DB 데이터가 'long_format_for_db_final' 변수에 저장되었습니다.
총 726개 레코드
컬럼: ['date', 'ticker', 'indicator', 'value']
Long Format DB 데이터 생성 완료


In [62]:
# -----------------------------
# 9단계 : Long Format DB 데이터 생성 (수정본)
# -----------------------------

table_name = "Korea_company_valuation_ver2"

# long_format_for_db_final 데이터를 DB에 저장
save_valuation_to_db(db_info, table_name, long_format_for_db_final)


저장할 데이터: ticker=A000660, 날짜=98개
기존 데이터: 98개 날짜
겹치는 날짜: 98개
새로운 날짜: 0개
기존 ticker A000660 데이터 삭제 완료
✅ 726건 데이터가 'Korea_company_valuation_ver2' 테이블에 저장되었습니다.
최종 확인: ticker A000660의 총 726건 데이터, 98개 날짜
